# Train Mountain General 9.6 km Residual U-Net

Train from the packaged four-domain LCP-canopy dataset and evaluate whether the residual U-Net improves mass-solver winds. Inputs are z_rel, dzdx, dzdy, canopy_cover, u_mass, and v_mass.

The default run now trains the all-domain LCP-canopy model after the Loveland, Keystone, and Breck/Tenmile holdout checks. This is the deployment-style checkpoint: it trains on all four terrain boxes and evaluates each HRRR and controlled source separately on the dataset test split.


In [ ]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import sys
import zipfile
from collections import Counter

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
LOCAL_ROOT = Path('/content/windninja_ml') if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path('/content/drive/MyDrive/windninja_ml') if IN_COLAB else Path.cwd() / 'ml/residual_unet/outputs/colab_local'
LOCAL_DATA_ROOT = Path('/content/data') if IN_COLAB else Path.cwd() / 'ml/residual_unet/data/processed'
ARTIFACT_ROOT = LOCAL_ROOT / 'artifacts'

GCP_PROJECT = os.environ.get('GCP_PROJECT', 'spring-nova-475120-r0')
GCS_BUCKET = os.environ.get('GCS_BUCKET', 'mwn-ml-general-9p6-spring-nova-475120-r0')
USE_GCS_ARTIFACTS = True
USE_DRIVE_RESULTS = True
SYNC_RESULTS_TO_GCS = True
FORCE_DOWNLOAD_CODE = True
FORCE_DOWNLOAD_DATA = False
FORCE_UNPACK_CODE = True
FORCE_UNPACK_DATA = False

DATASET_NAME = 'mountain_general_9p6_lcp_canopy_v1'
RUN_NAMES = [
    'mountain_general_9p6_lcp_canopy_v1',
]
# Completed holdout runs can be rerun by replacing RUN_NAMES with any of:
# mountain_general_9p6_lcp_canopy_holdout_loveland_v1
# mountain_general_9p6_lcp_canopy_holdout_keystone_v1
# mountain_general_9p6_lcp_canopy_holdout_breck_v1
RUN_NAME = RUN_NAMES[0]
CONFIG_NAME = f'{RUN_NAME}.yaml'

# Set SMOKE_TEST=True for a cheap first pass that verifies GCS download,
# local-disk unpacking, GPU training, evaluation, and result sync.
SMOKE_TEST = False
SMOKE_EPOCHS = 2
SMOKE_MAX_TRAIN_SAMPLES = 1024
SMOKE_MAX_VAL_SAMPLES = 256
SMOKE_MAX_EVAL_SAMPLES = 128

TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
NUM_WORKERS = 2
PREFETCH_FACTOR = 4
PROGRESS_EVERY = 50 if SMOKE_TEST else 100

DATASET_ZIP = ARTIFACT_ROOT / f'{DATASET_NAME}_dataset.zip'
CODE_ZIP = ARTIFACT_ROOT / 'residual_unet_code.zip'

REPO_DIR = LOCAL_ROOT / 'repo'
DATA_DIR = LOCAL_DATA_ROOT / DATASET_NAME
RESULT_ROOT = DRIVE_ROOT / 'results' if USE_DRIVE_RESULTS else LOCAL_ROOT / 'results'

HELDOUT_SOURCES = {
    'loveland': [
        'loveland_abasin_9p6_hrrr_lcp_canopy_v1',
        'loveland_abasin_9p6_controlled_lcp_canopy_9p6_15deg',
    ],
    'keystone': [
        'keystone_9p6_hrrr_lcp_canopy_v1',
        'keystone_9p6_controlled_lcp_canopy_9p6_15deg',
    ],
    'breck': [
        'breck_tenmile_9p6_hrrr_lcp_canopy_v1',
        'breck_tenmile_9p6_controlled_lcp_canopy_9p6_15deg',
    ],
}
ALL_DOMAIN_EVAL_SOURCES = [
    'berthoud_pass_hrrr_lcp_canopy_v1',
    'berthoud_pass_controlled_lcp_canopy_9p6_15deg',
    'breck_tenmile_9p6_hrrr_lcp_canopy_v1',
    'breck_tenmile_9p6_controlled_lcp_canopy_9p6_15deg',
    'keystone_9p6_hrrr_lcp_canopy_v1',
    'keystone_9p6_controlled_lcp_canopy_9p6_15deg',
    'loveland_abasin_9p6_hrrr_lcp_canopy_v1',
    'loveland_abasin_9p6_controlled_lcp_canopy_9p6_15deg',
]


def eval_label_for_run(run_name: str) -> str:
    if 'holdout_keystone' in run_name:
        return 'keystone'
    if 'holdout_breck' in run_name:
        return 'breck'
    if 'holdout_loveland' in run_name:
        return 'loveland'
    return 'all_domains'


def eval_sources_for_run(run_name: str) -> list[str]:
    label = eval_label_for_run(run_name)
    if label == 'all_domains':
        return ALL_DOMAIN_EVAL_SOURCES
    return HELDOUT_SOURCES[label]


def paths_for_run(run_name: str) -> dict[str, Path | str]:
    active_run_name = f'{run_name}_smoke' if SMOKE_TEST else run_name
    result_dir = RESULT_ROOT / active_run_name
    return {
        'active_run_name': active_run_name,
        'result_dir': result_dir,
        'checkpoint_dir': result_dir / 'checkpoints',
        'log_csv': result_dir / 'train_log.csv',
        'eval_root': result_dir / 'eval',
    }

RUN_PATHS = {run_name: paths_for_run(run_name) for run_name in RUN_NAMES}
ACTIVE_RUN_NAME = RUN_PATHS[RUN_NAME]['active_run_name']
RESULT_DIR = RUN_PATHS[RUN_NAME]['result_dir']
CHECKPOINT_DIR = RUN_PATHS[RUN_NAME]['checkpoint_dir']
LOG_CSV = RUN_PATHS[RUN_NAME]['log_csv']
EVAL_ROOT = RUN_PATHS[RUN_NAME]['eval_root']

print('IN_COLAB', IN_COLAB)
print('DATASET_NAME', DATASET_NAME)
print('RUN_NAMES')
for run_name in RUN_NAMES:
    print(' ', run_name, 'eval=', eval_label_for_run(run_name))
print('SMOKE_TEST', SMOKE_TEST)
print('DATASET_ZIP', DATASET_ZIP)
print('CODE_ZIP', CODE_ZIP)
print('RESULT_ROOT', RESULT_ROOT)



## Authenticate, Download, And Unpack

This cell mounts Drive only for persistent results, downloads ZIP artifacts from GCS directly to Colab local disk, and unpacks code/data onto local Colab disk. Keeping the large dataset off Drive avoids slow many-file Drive I/O during training.


In [ ]:
if IN_COLAB:
    from google.colab import auth, drive
    auth.authenticate_user()
    if USE_DRIVE_RESULTS:
        drive.mount('/content/drive')
    if USE_GCS_ARTIFACTS:
        subprocess.run(['gcloud', 'config', 'set', 'project', GCP_PROJECT], check=True)

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
for run_paths in RUN_PATHS.values():
    run_paths['result_dir'].mkdir(parents=True, exist_ok=True)


def copy_from_gcs_if_missing(blob_name: str, local_path: Path, *, force: bool = False) -> None:
    if local_path.exists() and not force:
        print(f'Found {local_path}')
        return
    if not USE_GCS_ARTIFACTS:
        raise FileNotFoundError(local_path)
    gcs_path = f'gs://{GCS_BUCKET}/drive_upload/{blob_name}'
    print(f'Copying {gcs_path} -> {local_path}')
    subprocess.run(['gcloud', 'storage', 'cp', gcs_path, str(local_path)], check=True)


def unpack_zip(zip_path: Path, dest_dir: Path, marker: Path, *, force: bool = False) -> None:
    if force and dest_dir.exists():
        print(f'Removing stale unpacked directory: {dest_dir}')
        shutil.rmtree(dest_dir)
    if marker.exists():
        print(f'Already unpacked: {marker}')
        return
    assert zip_path.exists(), zip_path
    print(f'Unpacking {zip_path} -> {dest_dir}')
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(dest_dir)

copy_from_gcs_if_missing('residual_unet_code.zip', CODE_ZIP, force=FORCE_DOWNLOAD_CODE)
copy_from_gcs_if_missing(f'{DATASET_NAME}_dataset.zip', DATASET_ZIP, force=FORCE_DOWNLOAD_DATA)

unpack_zip(CODE_ZIP, REPO_DIR, REPO_DIR / 'ml/residual_unet/train.py', force=FORCE_UNPACK_CODE)
unpack_zip(DATASET_ZIP, LOCAL_DATA_ROOT, DATA_DIR / 'manifest.csv', force=FORCE_UNPACK_DATA)

usage = shutil.disk_usage('/content' if IN_COLAB else Path.cwd())
print('Repo:', REPO_DIR)
print('Data:', DATA_DIR)
print('Artifacts:', ARTIFACT_ROOT)
print('Result root:', RESULT_ROOT)
print('Run result dirs:')
for run_name, run_paths in RUN_PATHS.items():
    print(' ', run_name, '->', run_paths['result_dir'])
print('Local disk free GB:', round(usage.free / 1024**3, 1))


## Install And Check Runtime


In [ ]:
requirements = REPO_DIR / 'ml/residual_unet/requirements.txt'
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
else:
    print('Skipping install outside Colab')

import torch
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    print('gpu_memory_gb', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


## Inspect Dataset And Split


In [ ]:
sys.path.insert(0, str(REPO_DIR))
from ml.residual_unet.config import load_config
from ml.residual_unet.dataset import filter_rows

summary = json.loads((DATA_DIR / 'dataset_summary.json').read_text())
rows = list(csv.DictReader((DATA_DIR / 'manifest.csv').open()))

print(json.dumps(summary, indent=2)[:4000])
print('total manifest rows:', len(rows))
print('by source:')
for source, count in Counter(row['source_dataset'] for row in rows).most_common():
    print(f'  {source}: {count}')

for run_name in RUN_NAMES:
    config_path = REPO_DIR / 'ml/residual_unet/configs' / f'{run_name}.yaml'
    config = load_config(config_path)
    data_cfg = config['data']
    train_excludes = [item.strip() for item in str(data_cfg.get('train_exclude_source_datasets', '')).split(',') if item.strip()]
    val_excludes = [item.strip() for item in str(data_cfg.get('val_exclude_source_datasets', '')).split(',') if item.strip()]
    train_rows = filter_rows(rows, 'train', exclude_source_datasets=train_excludes)
    val_rows = filter_rows(rows, 'val', exclude_source_datasets=val_excludes)
    planned_train_rows = train_rows[:SMOKE_MAX_TRAIN_SAMPLES] if SMOKE_TEST else train_rows
    planned_val_rows = val_rows[:SMOKE_MAX_VAL_SAMPLES] if SMOKE_TEST else val_rows

    print()
    print('run:', run_name)
    print('config:', config_path)
    print('training rows for this run:', len(train_rows))
    print('validation rows for this run:', len(val_rows))
    print('planned training rows:', len(planned_train_rows))
    print('planned validation rows:', len(planned_val_rows))
    print('held out from train/val:', train_excludes or 'none')


## Train Model Queue

The default run trains the final all-domain LCP-canopy model:

- `mountain_general_9p6_lcp_canopy_v1`

This is the likely deployment candidate because it trains on Berthoud, Breck/Tenmile, Keystone, and Loveland/A-Basin together. To rerun a terrain holdout, edit `RUN_NAMES` in the setup cell to one or more holdout config names.

For a cheap end-to-end GCS/Colab test, set `SMOKE_TEST = True` in the setup cell before running. Smoke mode uses separate result folders, two epochs, and capped train/validation/evaluation samples.


In [ ]:
import importlib
importlib.invalidate_caches()
for module_name in [
    'ml.residual_unet.config',
    'ml.residual_unet.dataset',
    'ml.residual_unet.train',
]:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

from ml.residual_unet.config import apply_overrides, load_config
import ml.residual_unet.train as train_module

for run_name in RUN_NAMES:
    run_paths = RUN_PATHS[run_name]
    active_run_name = run_paths['active_run_name']
    checkpoint_dir = run_paths['checkpoint_dir']
    log_csv = run_paths['log_csv']
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_csv.parent.mkdir(parents=True, exist_ok=True)

    config_path = REPO_DIR / 'ml/residual_unet/configs' / f'{run_name}.yaml'
    resume_path = checkpoint_dir / 'latest.pt'
    overrides = {
        'data.processed_dir': str(DATA_DIR),
        'data.batch_size': TRAIN_BATCH_SIZE,
        'data.num_workers': NUM_WORKERS,
        'data.prefetch_factor': PREFETCH_FACTOR,
        'data.pin_memory': True,
        'training.checkpoint_dir': str(checkpoint_dir),
        'training.log_csv': str(log_csv),
        'training.progress_every': PROGRESS_EVERY,
    }
    if SMOKE_TEST:
        overrides.update({
            'training.epochs': SMOKE_EPOCHS,
            'data.max_train_samples': SMOKE_MAX_TRAIN_SAMPLES,
            'data.max_val_samples': SMOKE_MAX_VAL_SAMPLES,
        })

    config = apply_overrides(load_config(config_path), overrides)
    resume = resume_path if resume_path.exists() else None

    print()
    print('=' * 80)
    print('direct train() call')
    print('run_name:', run_name)
    print('config:', config_path)
    print('active_run_name:', active_run_name)
    print('checkpoint_dir:', checkpoint_dir)
    print('log_csv:', log_csv)
    print('resume:', resume)
    print('overrides:', json.dumps(overrides, indent=2))
    train_module.train(config, resume=resume)


## Evaluate Held-Out Terrain

Run separate HRRR-only and controlled-only reports so the controlled matrix does not hide weather-case weakness.


In [ ]:
import importlib
importlib.invalidate_caches()
for module_name in [
    'ml.residual_unet.dataset',
    'ml.residual_unet.evaluate',
]:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

import ml.residual_unet.evaluate as evaluate_module

for run_name in RUN_NAMES:
    run_paths = RUN_PATHS[run_name]
    eval_label = eval_label_for_run(run_name)
    checkpoint = run_paths['checkpoint_dir'] / 'best.pt'
    assert checkpoint.exists(), checkpoint

    print()
    print('=' * 80)
    print('evaluating run:', run_name)
    print('eval label:', eval_label)
    print('checkpoint:', checkpoint)
    for source in eval_sources_for_run(run_name):
        out = run_paths['eval_root'] / source
        print('evaluating', source)
        metrics = evaluate_module.evaluate(
            checkpoint,
            DATA_DIR,
            out,
            split='test',
            source_datasets=[source],
            batch_size=EVAL_BATCH_SIZE,
            num_workers=NUM_WORKERS,
            prefetch_factor=PREFETCH_FACTOR,
            max_samples=SMOKE_MAX_EVAL_SAMPLES if SMOKE_TEST else None,
            max_figures=2 if SMOKE_TEST else 5,
        )
        print(source, json.dumps(metrics, indent=2))



## Package Results Back To GCS

The checkpoints, training log, metrics, sample CSVs, and figures are written under `MyDrive/windninja_ml/results/<active_run_name>/` by default. This final cell syncs that result directory back to the project bucket for local inspection and backup.


In [ ]:
for run_name in RUN_NAMES:
    run_paths = RUN_PATHS[run_name]
    result_dir = run_paths['result_dir']
    active_run_name = run_paths['active_run_name']
    print()
    print('=' * 80)
    print('Result files for', active_run_name)
    for path in sorted(result_dir.rglob('*')):
        if path.is_file():
            print(path.relative_to(result_dir))

    if SYNC_RESULTS_TO_GCS and IN_COLAB:
        subprocess.run(['gcloud', 'config', 'set', 'project', GCP_PROJECT], check=True)
        gcs_dir = f'gs://{GCS_BUCKET}/colab_results/{active_run_name}'
        subprocess.run(['gcloud', 'storage', 'rsync', '-r', str(result_dir), gcs_dir], check=True)
        subprocess.run(['gcloud', 'storage', 'ls', gcs_dir + '/'], check=True)
    else:
        print('Skipping GCS sync')


## Compare All Result Folders

Build a progression report across every result folder currently available in `MyDrive/windninja_ml/results`. This compares older holdout models, the all-domain model, and future reruns side by side by source/domain.


In [ ]:
import importlib
importlib.invalidate_caches()
if 'ml.residual_unet.compare_results' in sys.modules:
    importlib.reload(sys.modules['ml.residual_unet.compare_results'])

from ml.residual_unet.compare_results import compare_results

COMPARISON_DIR = RESULT_ROOT / '_comparison'
summary = compare_results(RESULT_ROOT, COMPARISON_DIR)
print('comparison rows:', len(summary['rows']))
print('comparison report:', COMPARISON_DIR / 'comparison_report.md')
print('run summaries:')
for row in summary['run_summaries']:
    print(
        row['run_name'],
        'sources=', row['source_count'],
        'ml_rmse=', round(row['ml_vector_rmse'], 3),
        'improvement=', round(row['vector_rmse_improvement_percent'], 1),
    )

if SYNC_RESULTS_TO_GCS and IN_COLAB:
    subprocess.run(['gcloud', 'config', 'set', 'project', GCP_PROJECT], check=True)
    gcs_dir = f'gs://{GCS_BUCKET}/colab_results/_comparison'
    subprocess.run(['gcloud', 'storage', 'rsync', '-r', str(COMPARISON_DIR), gcs_dir], check=True)
    subprocess.run(['gcloud', 'storage', 'ls', gcs_dir + '/'], check=True)
else:
    print('Skipping GCS comparison sync')
